# Image Classification with Deep Learning and CNNs

## Dataset Choice: Fashion-MNIST


## Imports


In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import fashion_mnist
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow version:", tf.__version__)


# Data Preparation
- Load Fashion-MNIST
- Display 10 sample images with labels
- Normalize pixels to 0–1
- Create train, validation, and test sets


In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = fashion_mnist.load_data()
class_names = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]
print("Original training shape:", x_train_full.shape)
print("Test shape:", x_test.shape)


In [ ]:
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train_full[i], cmap="gray")
    plt.title(class_names[y_train_full[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()


## Normalization and split


In [ ]:
x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.20, random_state=SEED, stratify=y_train_full
)
print("Train:", x_train.shape)
print("Validation:", x_val.shape)
print("Test:", x_test.shape)
print("Pixel range:", x_train.min(), "to", x_train.max())


# Baseline Neural Network 
Architecture: **Flatten → Dense(ReLU) → Dense(ReLU) → Softmax**.


In [ ]:
dense_model = keras.Sequential(
    [
        layers.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ],
    name="Dense_Neural_Network",
)

dense_model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
dense_model.summary()


In [ ]:
dense_history = dense_model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=15,
    batch_size=128,
    verbose=1,
)


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(dense_history.history["accuracy"], label="Training Accuracy")
plt.plot(dense_history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Dense Model Accuracy")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(dense_history.history["loss"], label="Training Loss")
plt.plot(dense_history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Dense Model Loss")
plt.legend()
plt.grid(True)
plt.show()


#  CNN Model
The CNN uses at least two Conv2D + MaxPooling blocks, plus Batch Normalization and Dropout to reduce overfitting.


In [ ]:
x_train_cnn = np.expand_dims(x_train, -1)
x_val_cnn = np.expand_dims(x_val, -1)
x_test_cnn = np.expand_dims(x_test, -1)

cnn_model = keras.Sequential(
    [
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(10, activation="softmax"),
    ],
    name="CNN_Model",
)

cnn_model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
cnn_model.summary()


In [ ]:
cnn_history = cnn_model.fit(
    x_train_cnn,
    y_train,
    validation_data=(x_val_cnn, y_val),
    epochs=15,
    batch_size=128,
    verbose=1,
)


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(cnn_history.history["accuracy"], label="Training Accuracy")
plt.plot(cnn_history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN Accuracy")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(cnn_history.history["loss"], label="Training Loss")
plt.plot(cnn_history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CNN Loss")
plt.legend()
plt.grid(True)
plt.show()


# Evaluation and Comparison


In [ ]:
dense_test_loss, dense_test_acc = dense_model.evaluate(x_test, y_test, verbose=0)
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test_cnn, y_test, verbose=0)

print(f"Dense Test Accuracy: {dense_test_acc:.4f}")
print(f"CNN Test Accuracy:   {cnn_test_acc:.4f}")


In [ ]:
dense_trainable = int(np.sum([np.prod(v.shape) for v in dense_model.trainable_weights]))
cnn_trainable = int(np.sum([np.prod(v.shape) for v in cnn_model.trainable_weights]))

comparison_df = pd.DataFrame(
    {
        "Model": ["Dense Neural Network", "CNN"],
        "Trainable Parameters": [dense_trainable, cnn_trainable],
        "Test Loss": [dense_test_loss, cnn_test_loss],
        "Test Accuracy": [dense_test_acc, cnn_test_acc],
    }
)
comparison_df


## Confusion Matrix for CNN


In [ ]:
cnn_probs = cnn_model.predict(x_test_cnn, verbose=0)
cnn_preds = np.argmax(cnn_probs, axis=1)
cm = confusion_matrix(y_test, cnn_preds)
fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False
)
plt.title("CNN Confusion Matrix")
plt.tight_layout()
plt.show()


## Five Misclassified Images


In [ ]:
wrong = np.where(cnn_preds != y_test)[0]
print("Total misclassified images:", len(wrong))
plt.figure(figsize=(15, 4))
for i, idx in enumerate(wrong[:5]):
    pred = cnn_preds[idx]
    actual = y_test[idx]
    confidence = cnn_probs[idx][pred]
    plt.subplot(1, 5, i + 1)
    plt.imshow(x_test[idx], cmap="gray")
    plt.title(
        f"Actual: {class_names[actual]}\nPred: {class_names[pred]}\nConf: {confidence:.2%}"
    )
    plt.axis("off")
plt.tight_layout()
plt.show()


### Why might the CNN misclassify these images?
Fashion-MNIST contains several visually similar classes. Shirts, T-shirts, pullovers, and coats can have very similar silhouettes in small 28×28 grayscale images. The low resolution also removes details such as color, texture, buttons, and logos. Because of these ambiguities, even a well-trained CNN can confuse visually related categories.


#  Reflection

The CNN generally performs better than the fully-connected dense network because it is designed to exploit the spatial structure of images. A dense network first flattens each 28×28 image into a one-dimensional vector. After flattening, the model no longer explicitly preserves the relationship between neighboring pixels, so it must learn image patterns using fully connected weights. A CNN instead applies small filters to local image regions. These filters can detect edges, curves, corners, textures, and shape patterns. Max-pooling reduces the spatial dimensions while preserving important features. CNNs also use parameter sharing, meaning the same filter is reused in many image locations. This makes them more efficient at learning visual features than ordinary dense layers.

Overfitting can be observed by comparing training and validation curves. If training accuracy keeps increasing while validation accuracy stops improving or falls, the model may be memorizing the training data. A similar pattern occurs when training loss keeps decreasing but validation loss starts increasing. In this CNN, Batch Normalization helps stabilize the learning process, while Dropout randomly disables some activations during training. Dropout reduces dependence on specific neurons and can improve generalization. The actual amount of overfitting should be judged from the plots generated after running the notebook.

Several techniques could be used to improve accuracy further. Data augmentation could produce slightly changed versions of training images using small translations, rotations, or zoom operations. The CNN could also be made deeper by adding more convolutional blocks or by tuning the number of filters. Hyperparameters such as learning rate, batch size, kernel size, dropout rate, and number of epochs could be optimized. Early stopping could halt training when validation loss no longer improves, and a learning-rate scheduler could reduce the learning rate during later epochs. Another option would be to experiment with transfer learning after adapting the grayscale images to a suitable pretrained image model. Finally, an ensemble of multiple models could improve robustness by combining predictions from models that make different errors.


# Final Summary
This notebook includes all required assignment sections: data preparation, dense baseline, CNN, training curves, test accuracy, trainable parameter comparison, confusion matrix, five misclassified examples, and a 300–400 word reflection.
